In [7]:
import transformers
import datasets
from IPython.display import Image, display
import random
import pydicom

transformers.__version__

'4.57.1'

In [8]:
alias = 'aehrc/cxrmate-2'

model = transformers.AutoModelForCausalLM.from_pretrained(alias, trust_remote_code=True).to(device='cuda')
model.eval()
generation_config = transformers.GenerationConfig.from_pretrained(alias, trust_remote_code=True)
processor = transformers.AutoProcessor.from_pretrained(alias, trust_remote_code=True)

Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 13.08it/s]
A new version of the following files was downloaded from https://huggingface.co/aehrc/cxrmate-2:
- dataset.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/aehrc/cxrmate-2:
- processing_cxrmate2.py
- dataset.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [9]:
url = 'https://prod-images-static.radiopaedia.org/images/220869/76052f7902246ff862f52f5d3cd9cd_big_gallery.jpg'
display(Image(url=url))
processed = processor(images=url)
processed = processed.to(device='cuda')
generated_ids = model.generate(**processed, generation_config=generation_config)
findings, impression = processor.split_and_decode_sections(generated_ids) 
print(f'Findings:\t{findings[0]}\nImpression:\t{impression[0]}')

Findings:	Heart size, mediastinal and hilar contours are normal. Lungs are well expanded and clear. There are no pleural effusions or acute skeletal findings.
Impression:	No radiographic evidence of pneumonia.


In [13]:
dcm_path = [
    '/datasets/work/hb-mlaifsp-mm/work/repositories/25_cxrmate2/work/data/physionet.org/files/mimic-cxr/2.0.0/files/p12/p12000264/s55271473/522f9570-7cb12ecb-6327c8b8-b248b4be-58bb3dfd.dcm',
    '/datasets/work/hb-mlaifsp-mm/work/repositories/25_cxrmate2/work/data/physionet.org/files/mimic-cxr/2.0.0/files/p12/p12000264/s55271473/d0b61aff-f64c4ecf-85fae310-43668cf1-0c1d4c2d.dcm',
]
views = [pydicom.dcmread(p).ViewPosition for p in dcm_path]
print(f'Views: {views}')

processed = processor(images=dcm_path, views=views)
display(processed['pixel_values'].shape)
processed = processed.to(device='cuda')
generated_ids = model.generate(**processed, generation_config=generation_config)
findings, impression = processor.split_and_decode_sections(generated_ids) 
print(f'Findings:\t{findings[0]}\nImpression:\t{impression[0]}')

Views: ['PA', 'LATERAL']


torch.Size([1, 2, 3, 518, 518])

Findings:	Heart is upper limits of normal in size. Aorta is tortuous. Lungs are clear except for minimal scarring at the left base. There are no pleural effusions or acute skeletal findings.
Impression:	No radiographic evidence of pneumonia.


In [11]:
# Run prepare_chexpert_plus.py or prepare_mimic_cxr_jpg.py or prepare_rexgradient.py to create dataset:
test_set = datasets.load_from_disk('/scratch3/nic261/database/cxrmate2/rexgradient_160k_dataset')['test']
display(test_set)

# Wrap dataset to get priors:
test_set = processor.wrap_dataset(test_set)

Dataset({
    features: ['id', 'AccessionNumber', 'study_id', 'PatientSex', 'PatientAge', 'study_datetime', 'technique', 'indication', 'comparison', 'findings', 'impression', 'subject_id', 'prior_study_ids', 'prior_study_datetimes', 'demographics', 'images'],
    num_rows: 10000
})

In [12]:
random_idx = random.randint(0, len(test_set) - 1)
random_idx = 6654
example = test_set[random_idx]

processed = processor(
    images=example['images'],  # This includes both the current and prior images.
    image_datetime=example['image_datetime'],  # This includes the datetimes for both the current and prior images.
    views=example['views'],  # This includes both the current and prior views.
    indication=example.get('indication', None),
    history=example.get('history', None),
    comparison=example.get('comparison', None),
    technique=example.get('technique', None),
    study_datetime=example.get('study_datetime', None),
    prior_findings=example.get('prior_findings', None),
    prior_impression=example.get('prior_impression', None),
    prior_study_datetime=example.get('prior_study_datetime', None),
)
processed = processed.to(device='cuda')
generated_ids = model.generate(**processed, generation_config=generation_config)
findings, impression = processor.split_and_decode_sections(generated_ids) 
print(f'Findings:\t{findings[0]}\nImpression:\t{impression[0]}')
print(f'\nGT Findings:\t{example["findings"]}\nGT Impression:\t{example["impression"]}')

Findings:	Patient is rotated. Hazy opacity at the left lung base, consistent with pleural effusion and airspace opacity in the left mid lower lung zone. Minimal patchy opacity in the right lung base. Unchanged heart size and mediastinal contours allowing for patient rotation. No pneumothorax. No pulmonary edema. No acute osseous abnormalities are seen.
Impression:	1. Hazy left lung base opacity consistent with pleural effusion and airspace disease, pattern typical of COVID pneumonia. 2. Patchy right lung base opacity may represent atelectasis or pneumonia.

GT Findings:	Low lung volumes limit assessment. Left pleural effusion which has increased from radiographs last month. Associated increased opacity throughout the left hemithorax, also progressed. Heart appears enlarged, however mediastinal contours are not well assessed given positioning. No pneumothorax. No focal right lung abnormality. The bones are under mineralized. Left rib fractures, likely subacute.
GT Impression:	1. Worseni